# 08 - Churn Prediction
Train LR, RF, and XGBoost models to predict customer churn.

In [1]:
import pandas as pd, numpy as np, joblib, os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix, roc_curve
import plotly.express as px, plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings; warnings.filterwarnings('ignore')
SEED=42; np.random.seed(SEED)
PRIMARY='#635BFF'; RISK='#E74C3C'; SAFE='#27AE60'; NEUTRAL='#3498DB'; WARNING='#F39C12'; TEMPLATE='plotly_white'


In [2]:
df = pd.read_csv('data/processed/customer_clean.csv')
df = df[[col for col in df.columns if 'Naive_Bayes' not in col]]
df['churn_label'] = (df['Attrition_Flag'] == 'Attrited Customer').astype(int)
print(f"Shape: {df.shape}")
print(f"Churn distribution:\n{df['churn_label'].value_counts()}")


Shape: (10127, 22)
Churn distribution:
churn_label
0    8500
1    1627
Name: count, dtype: int64


In [3]:
# Encode categoricals
cat_cols = ['Income_Category','Card_Category','Gender','Education_Level','Marital_Status']
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

numeric_features = ['Customer_Age','Dependent_count','Months_on_book',
    'Months_Inactive_12_mon','Contacts_Count_12_mon','Credit_Limit',
    'Total_Revolving_Bal','Avg_Open_To_Buy','Total_Amt_Chng_Q4_Q1',
    'Total_Trans_Amt','Total_Trans_Ct','Total_Ct_Chng_Q4_Q1','Avg_Utilization_Ratio']
all_features = numeric_features + cat_cols

X = df[all_features]
y = df['churn_label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


Train: (8101, 18), Test: (2026, 18)


In [4]:
# Train models
models = {}
# Logistic Regression
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED, solver='lbfgs')
lr.fit(X_train, y_train)
models['Logistic Regression'] = lr

# Random Forest
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
models['Random Forest'] = rf

# XGBoost
scale_pos = (y==0).sum() / (y==1).sum()
xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05,
                     scale_pos_weight=scale_pos, eval_metric='auc',
                     random_state=SEED, use_label_encoder=False)
xgb.fit(X_train, y_train)
models['XGBoost'] = xgb
print("All models trained.")


All models trained.


In [5]:
# Evaluate models
results = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    results.append({'Model': name, 'Accuracy': acc, 'Precision': prec,
                    'Recall': rec, 'F1': f1, 'ROC-AUC': auc})
    print(f"\n{name}:")
    print(classification_report(y_test, y_pred))

results_df = pd.DataFrame(results)
print("\n=== Model Comparison ===")
print(results_df.round(4))
best_model_name = results_df.loc[results_df['ROC-AUC'].idxmax(), 'Model']
print(f"\nBest model by ROC-AUC: {best_model_name}")



Logistic Regression:
              precision    recall  f1-score   support

           0       0.95      0.82      0.88      1701
           1       0.45      0.78      0.57       325

    accuracy                           0.81      2026
   macro avg       0.70      0.80      0.73      2026
weighted avg       0.87      0.81      0.83      2026




Random Forest:
              precision    recall  f1-score   support

           0       0.95      0.99      0.97      1701
           1       0.92      0.74      0.82       325

    accuracy                           0.95      2026
   macro avg       0.94      0.87      0.90      2026
weighted avg       0.95      0.95      0.95      2026




XGBoost:
              precision    recall  f1-score   support

           0       0.98      0.97      0.98      1701
           1       0.86      0.90      0.88       325

    accuracy                           0.96      2026
   macro avg       0.92      0.94      0.93      2026
weighted avg       0.96      0.96      0.96      2026


=== Model Comparison ===
                 Model  Accuracy  Precision  Recall      F1  ROC-AUC
0  Logistic Regression    0.8124     0.4512  0.7815  0.5721   0.8900
1        Random Forest    0.9492     0.9237  0.7446  0.8245   0.9859
2              XGBoost    0.9605     0.8614  0.8985  0.8795   0.9906

Best model by ROC-AUC: XGBoost


In [6]:
fig = go.Figure()
colors = [PRIMARY, SAFE, RISK]
for i, (name, model) in enumerate(models.items()):
    y_proba = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f"{name} (AUC={auc:.3f})", line=dict(color=colors[i])))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], name='Random', line=dict(dash='dash', color='gray')))
fig.update_layout(title='ROC Curves', xaxis_title='FPR', yaxis_title='TPR', template=TEMPLATE)
fig.show()


In [7]:
best_model = models[best_model_name]
cm = confusion_matrix(y_test, best_model.predict(X_test))
fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
                labels=dict(x='Predicted', y='Actual'),
                x=['Existing','Attrited'], y=['Existing','Attrited'],
                template=TEMPLATE, title=f'Confusion Matrix - {best_model_name}')
fig.show()


In [8]:
for name, model in [('Random Forest', rf), ('XGBoost', xgb)]:
    imp = pd.DataFrame({'feature': all_features, 'importance': model.feature_importances_})
    imp = imp.sort_values('importance', ascending=True).tail(15)
    fig = px.bar(imp, y='feature', x='importance', orientation='h',
                 color_discrete_sequence=[PRIMARY if name=='Random Forest' else RISK],
                 template=TEMPLATE, title=f'Feature Importance - {name} (Top 15)')
    fig.show()


In [9]:
xgb_proba = xgb.predict_proba(X[all_features])[:,1]
fig = px.histogram(x=xgb_proba, nbins=50, color_discrete_sequence=[RISK],
                   template=TEMPLATE, title='Churn Probability Distribution (XGBoost)',
                   labels={'x':'Probability','y':'Count'})
fig.show()


In [10]:
df_orig = pd.read_csv('data/processed/customer_clean.csv')
df_orig['churn_label'] = (df_orig['Attrition_Flag'] == 'Attrited Customer').astype(int)

for col, title in [('Card_Category','Churn Rate by Card Category'),
                   ('Income_Category','Churn Rate by Income Category'),
                   ('Gender','Churn Rate by Gender')]:
    cr = df_orig.groupby(col)['churn_label'].mean().reset_index()
    cr.columns = [col, 'churn_rate']
    fig = px.bar(cr, x=col, y='churn_rate', color_discrete_sequence=[RISK],
                 template=TEMPLATE, title=title)
    fig.show()

# Months_Inactive binned vs churn rate
df_orig['inactive_bin'] = pd.cut(df_orig['Months_Inactive_12_mon'], bins=7)
bin_churn = df_orig.groupby('inactive_bin', observed=True)['churn_label'].mean().reset_index()
bin_churn['inactive_bin'] = bin_churn['inactive_bin'].astype(str)
fig = px.line(bin_churn, x='inactive_bin', y='churn_label', markers=True,
              color_discrete_sequence=[RISK], template=TEMPLATE,
              title='Churn Rate by Months Inactive (binned)')
fig.show()


In [11]:
# Add churn_probability and save
df['churn_probability'] = xgb_proba
os.makedirs('models/churn', exist_ok=True)
joblib.dump(lr, 'models/churn/logistic_regression.pkl')
joblib.dump(rf, 'models/churn/random_forest.pkl')
joblib.dump(xgb, 'models/churn/xgboost_model.pkl')
print("Saved all churn models")

save_cols = ['CLIENTNUM','churn_probability'] + all_features
df[save_cols].to_csv('data/processed/churn_predictions.csv', index=False)
print(f"Saved churn_predictions.csv - shape: {df[save_cols].shape}")


Saved all churn models
Saved churn_predictions.csv - shape: (10127, 20)
